In [1]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-haiku-4-5"

In [ ]:
from datetime import datetime
from anthropic.types import ToolParam


def get_current_datetime(date_format="%Y-%m-%d %H:%M:%S"):
    if not date_format:
        raise ValueError("date_format cannot be empty")
    return datetime.now().strftime(date_format)


# use Claude to generate the description for you
get_current_datetime_schema = ToolParam(
    {
        "name": "get_current_datetime",
        "description": "Get the current date and time, formatted according to a strftime-style format string. Raises an error if an empty format string is given.",
        "input_schema": {
            "type": "object",
            "properties": {
                "date_format": {
                    "type": "string",
                    "description": "A Python strftime format string used to format the current datetime (e.g. '%Y-%m-%d %H:%M:%S', '%d/%m/%Y').",
                    "default": "%Y-%m-%d %H:%M:%S",
                }
            },
            "required": [],
        },
    }
)

In [ ]:
messages = []

messages.append(
    {"role": "user", "content": "What is the exact time, formatted as HH:MM:SS?"}
)

response = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=messages,
    tools=[get_current_datetime_schema],
)

messages.append({"role": "assistant", "content": response.content})

In [ ]:
input = response.content[0].input  # type: ignore
result = get_current_datetime(**input)  # type: ignore

messages.append(
    {
        "role": "user",
        "content": [
            # the Tool Result Block
            {
                "tool_use_id": response.content[0].id,  # type: ignore
                "type": "tool_result",  # <--
                "content": result,
                "is_error": False,
            }
        ],
    }
)

messages

[{'role': 'user', 'content': 'What is the exact time, formatted as HH:MM:SS?'},
 {'role': 'assistant',
  'content': [ToolUseBlock(id='toolu_01EwsAtnaXUHJBgxVGLMyznM', caller=DirectCaller(type='direct'), input={'date_format': '%H:%M:%S'}, name='get_current_datetime', type='tool_use')]},
 {'role': 'user',
  'content': [{'type': 'tool_result',
    'tool_use_id': 'toolu_01EwsAtnaXUHJBgxVGLMyznM',
    'content': '11:46:33',
    'is_error': False}]}]

In [5]:
client.messages.create(
    model=model,
    max_tokens=1000,
    messages=messages,
    tools=[get_current_datetime_schema] # always include if tools are involved in the conversation
)

Message(id='msg_011CcrGfU77XVouuyPTrX7bm', container=None, content=[TextBlock(citations=None, text='The exact time is **11:46:33**.', type='text')], model='claude-haiku-4-5-20251001', role='assistant', stop_details=None, stop_reason='end_turn', stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', input_tokens=735, output_tokens=14, output_tokens_details=None, server_tool_use=None, service_tier='standard'))